# 병렬화 워크플로 — Parallelization Workflows

**Skilljar Lesson 02 대응**

이 노트북에서 다루는 내용:
1. 순차 실행 vs 병렬 실행 성능 비교
2. `AsyncAnthropic` + `asyncio.gather`를 사용한 병렬 LLM 호출
3. 다관점 분석 (Sectioning) 패턴
4. 결과 종합 (Aggregation) 단계

In [ ]:
# ── Setup ──────────────────────────────────────────────
import anthropic
import asyncio
import time
from dotenv import load_dotenv

load_dotenv()

# 동기 클라이언트 (순차 실행용)
client = anthropic.Anthropic()
# 비동기 클라이언트 (병렬 실행용)
async_client = anthropic.AsyncAnthropic()

MODEL = "claude-haiku-4-5"

## §1. 분석 대상 코드와 관점 정의

하나의 코드를 보안/성능/가독성 세 가지 관점에서 동시에 분석합니다.

In [ ]:
code_to_review = """
def process_user_data(user_input):
    query = f"SELECT * FROM users WHERE name = '{user_input}'"
    result = db.execute(query)
    data = [row for row in result]
    return data
"""

perspectives = [
    {
        "name": "보안 분석",
        "system": "You are a security expert. Analyze the code for "
                  "security vulnerabilities. Be specific about risks "
                  "and provide fixes. Respond in Korean."
    },
    {
        "name": "성능 분석",
        "system": "You are a performance expert. Analyze the code for "
                  "performance issues and suggest optimizations. "
                  "Respond in Korean."
    },
    {
        "name": "가독성 분석",
        "system": "You are a code reviewer. Analyze the code for "
                  "readability and maintainability issues. "
                  "Respond in Korean."
    },
]

print(f"분석 관점 {len(perspectives)}개 정의 완료")

## §2. 순차 실행 (비교 기준)

In [ ]:
start = time.time()
sequential_results = []

for p in perspectives:
    response = client.messages.create(
        model=MODEL,
        max_tokens=1024,
        system=p["system"],
        messages=[{"role": "user", "content": f"Analyze this code:\n{code_to_review}"}]
    )
    sequential_results.append({"name": p["name"], "analysis": response.content[0].text})
    print(f"  ✅ {p['name']} 완료")

seq_time = time.time() - start
print(f"\n순차 실행 시간: {seq_time:.1f}초")

## §3. 병렬 실행 (asyncio.gather)

In [ ]:
async def analyze_with_perspective(perspective, code):
    """단일 관점으로 코드를 비동기 분석한다."""
    response = await async_client.messages.create(
        model=MODEL,
        max_tokens=1024,
        system=perspective["system"],
        messages=[{"role": "user", "content": f"Analyze this code:\n{code}"}]
    )
    return {
        "name": perspective["name"],
        "analysis": response.content[0].text
    }


async def parallel_analysis(code, perspectives):
    """여러 관점의 분석을 동시에 실행한다."""
    tasks = [analyze_with_perspective(p, code) for p in perspectives]
    results = await asyncio.gather(*tasks)
    return results

In [ ]:
start = time.time()
parallel_results = await parallel_analysis(code_to_review, perspectives)
par_time = time.time() - start

print(f"병렬 실행 시간: {par_time:.1f}초")
print(f"속도 향상: {seq_time / par_time:.1f}x ({(1 - par_time/seq_time)*100:.0f}% 단축)")

## §4. 결과 비교 및 종합

In [ ]:
for r in parallel_results:
    print(f"\n{'='*50}")
    print(f"📋 {r['name']}")
    print(f"{'='*50}")
    print(r["analysis"])

In [ ]:
# 종합 보고서 생성
combined = "\n\n".join(
    f"[{a['name']}]\n{a['analysis']}" for a in parallel_results
)

summary_response = client.messages.create(
    model=MODEL,
    max_tokens=2048,
    system=("You are a senior code reviewer. Synthesize the following "
            "multi-perspective analyses into a unified code review report "
            "with prioritized action items. Respond in Korean."),
    messages=[{
        "role": "user",
        "content": f"다관점 분석 결과를 종합해주세요:\n\n{combined}"
    }]
)

print("\n📊 종합 보고서")
print("=" * 50)
print(summary_response.content[0].text)

## §5. 핵심 정리

- **`AsyncAnthropic`**: 비동기 API 클라이언트
- **`asyncio.gather`**: 여러 코루틴을 동시에 실행하고 모든 결과를 수집
- **전체 시간 = max(개별 시간)**: 가장 느린 호출이 전체 시간을 결정
- **종합 단계**: 개별 분석을 하나의 보고서로 합치는 추가 LLM 호출
- **다음 노트북**: `S8_03_chaining.ipynb`에서 체이닝 워크플로를 구현한다